# MCP + LangGraph 핸즈온 튜토리얼

- 작성자: [테디노트](https://youtube.com/c/teddynote)
- 강의: [패스트캠퍼스 RAG 비법노트](https://fastcampus.co.kr/data_online_teddy)

**참고자료**
- https://modelcontextprotocol.io/introduction
- https://github.com/langchain-ai/langchain-mcp-adapters

## 환경설정

아래 설치 방법을 참고하여 `uv` 를 설치합니다.

**uv 설치 방법**

```bash
# macOS/Linux
curl -LsSf https://astral.sh/uv/install.sh | sh

# Windows (PowerShell)
irm https://astral.sh/uv/install.ps1 | iex
```

**의존성 설치**

```bash
uv pip install -r requirements.txt
```

환경변수를 가져옵니다.

In [1]:
from dotenv import load_dotenv
from langchain_teddynote import logging

load_dotenv(override=True)
logging.langsmith("LangGraph MCP Adapters HandsOn")

LangSmith 추적을 시작합니다.
[프로젝트명]
LangGraph MCP Adapters HandsOn


## MultiServerMCPClient

사전에 `mcp_server_remote.py` 를 실행해둡니다. 터미널을 열고 가상환경이 활성화 되어 있는 상태에서 서버를 실행해 주세요.

> 명령어
```bash
source .venv/bin/activate
python mcp_server_remote.py
```

`async with` 로 일시적인 Session 연결을 생성 후 해제

In [5]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langchain_teddynote.messages import ainvoke_graph, astream_graph
from langchain_anthropic import ChatAnthropic
from langchain_google_genai import ChatGoogleGenerativeAI

# model = ChatAnthropic(
#     model_name="claude-3-7-sonnet-latest", temperature=0, max_tokens=20000
# )
model = ChatGoogleGenerativeAI(model="gemini-2.0-flash-exp")

async with MultiServerMCPClient(
    {
        "weather": {
            # 서버의 포트와 일치해야 합니다.(8005번 포트)
            "url": "http://localhost:8005/sse",
            "transport": "sse",
        }
    }
) as client:
    print(client.get_tools())
    agent = create_react_agent(model, client.get_tools())
    answer = await astream_graph(agent, {"messages": "서울의 날씨는 어떠니?"})

[StructuredTool(name='get_weather', description='\n    Get current weather information for the specified location.\n\n    This function simulates a weather service by returning a fixed response.\n    In a production environment, this would connect to a real weather API.\n\n    Args:\n        location (str): The name of the location (city, region, etc.) to get weather for\n\n    Returns:\n        str: A string containing the weather information for the specified location\n    ', args_schema={'properties': {'location': {'title': 'Location', 'type': 'string'}}, 'required': ['location'], 'title': 'get_weatherArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x7f8c89d0cf40>)]

🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
It's always Sunny in 서울
🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - 

다음의 경우에는 session 이 닫혔기 때문에 도구에 접근할 수 없는 것을 확인할 수 있습니다.

In [6]:
await astream_graph(agent, {"messages": "서울의 날씨는 어떠니?"})


🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
Error: ClosedResourceError()
 Please fix your mistakes.
🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
죄송합니다. 서울의 날씨 정보를 가져오는 데 문제가 발생했습니다. 다시 시도해 주시겠습니까?

{'node': 'agent',
 'content': AIMessageChunk(content='도해 주시겠습니까?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash-exp', 'safety_ratings': []}, id='run-f87af7a9-967d-4df4-b62a-fe70bf6b4ce7', usage_metadata={'input_tokens': -50, 'output_tokens': 29, 'total_tokens': -21, 'input_token_details': {'cache_read': 0}}),
 'metadata': {'langgraph_step': 3,
  'langgraph_node': 'agent',
  'langgraph_triggers': ('branch:to:agent', 'start:agent', 'tools'),
  'langgraph_path': ('__pregel_pull', 'agent'),
  'langgraph_checkpoint_ns': 'agent:298235cc-30e3-02c4-a390-a82437ccbd0a',
  'checkpoint_ns': 'agent:298235cc-30e3-02c4-a390-a82437ccbd0a',
  'ls_provider': 'google_genai',
  'ls_model_name': 'models/gemini-2.0-flash-exp',
  'ls_model_type': 'chat',
  'ls_temperature': 0.7}}

이제 그럼 Async Session 을 유지하며 도구에 접근하는 방식으로 변경해 보겠습니다.

In [7]:
# 1. 클라이언트 생성
client = MultiServerMCPClient(
    {
        "weather": {
            "url": "http://localhost:8005/sse",
            "transport": "sse",
        }
    }
)


# 2. 명시적으로 연결 초기화 (이 부분이 필요함)
# 초기화
await client.__aenter__()

# 이제 도구가 로드됨
print(client.get_tools())  # 도구가 표시됨

[StructuredTool(name='get_weather', description='\n    Get current weather information for the specified location.\n\n    This function simulates a weather service by returning a fixed response.\n    In a production environment, this would connect to a real weather API.\n\n    Args:\n        location (str): The name of the location (city, region, etc.) to get weather for\n\n    Returns:\n        str: A string containing the weather information for the specified location\n    ', args_schema={'properties': {'location': {'title': 'Location', 'type': 'string'}}, 'required': ['location'], 'title': 'get_weatherArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x7f8c89d9b560>)]


langgraph 의 에이전트를 생성합니다.

In [8]:
# 에이전트 생성
agent = create_react_agent(model, client.get_tools())

그래프를 실행하여 결과를 확인합니다.

In [9]:
await astream_graph(agent, {"messages": "서울의 날씨는 어떠니?"})


🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
It's always Sunny in 서울
🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
서울은 항상 화창합니다.

{'node': 'agent',
 'content': AIMessageChunk(content=' 항상 화창합니다.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash-exp', 'safety_ratings': []}, id='run-13c4b388-b410-4576-b3d9-3389e3ffb733', usage_metadata={'input_tokens': -51, 'output_tokens': 8, 'total_tokens': -43, 'input_token_details': {'cache_read': 0}}),
 'metadata': {'langgraph_step': 3,
  'langgraph_node': 'agent',
  'langgraph_triggers': ('branch:to:agent', 'start:agent', 'tools'),
  'langgraph_path': ('__pregel_pull', 'agent'),
  'langgraph_checkpoint_ns': 'agent:7ee5a76e-83b9-d493-87ce-3acbc0416508',
  'checkpoint_ns': 'agent:7ee5a76e-83b9-d493-87ce-3acbc0416508',
  'ls_provider': 'google_genai',
  'ls_model_name': 'models/gemini-2.0-flash-exp',
  'ls_model_type': 'chat',
  'ls_temperature': 0.7}}

## Stdio 통신 방식

Stdio 통신 방식은 로컬 환경에서 사용하기 위해 사용합니다.

- 통신을 위해 표준 입력/출력 사용

참고: 아래의 python 경로는 수정하세요!

In [16]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langgraph.prebuilt import create_react_agent
from langchain_mcp_adapters.tools import load_mcp_tools
from langchain_anthropic import ChatAnthropic

# Anthropic의 Claude 모델 초기화
# model = ChatAnthropic(
#     model_name="claude-3-7-sonnet-latest", temperature=0, max_tokens=20000
# )
modle = ChatGoogleGenerativeAI(model="gemini-2.0-flash-exp")

# StdIO 서버 파라미터 설정
# - command: Python 인터프리터 경로
# - args: 실행할 MCP 서버 스크립트
server_params = StdioServerParameters(
    command="./.venv/bin/python",
    args=["mcp_server_local.py"],
)

# StdIO 클라이언트를 사용하여 서버와 통신
async with stdio_client(server_params) as (read, write):
    # 클라이언트 세션 생성
    async with ClientSession(read, write) as session:
        # 연결 초기화
        await session.initialize()

        # MCP 도구 로드
        tools = await load_mcp_tools(session)
        print(tools)

        # 에이전트 생성
        agent = create_react_agent(model, tools)

        # 에이전트 응답 스트리밍
        await astream_graph(agent, {"messages": "서울의 날씨는 어떠니?"})

[StructuredTool(name='get_weather', description='\n    Get current weather information for the specified location.\n\n    This function simulates a weather service by returning a fixed response.\n    In a production environment, this would connect to a real weather API.\n\n    Args:\n        location (str): The name of the location (city, region, etc.) to get weather for\n\n    Returns:\n        str: A string containing the weather information for the specified location\n    ', args_schema={'properties': {'location': {'title': 'Location', 'type': 'string'}}, 'required': ['location'], 'title': 'get_weatherArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x7f8c89d43560>)]

🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
It's always Sunny in 서울
🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - 

## RAG 를 구축한 MCP 서버 사용

- 파일: `mcp_server_rag.py`

사전에 langchain 으로 구축한 `mcp_server_rag.py` 파일을 사용합니다.

stdio 통신 방식으로 도구에 대한 정보를 가져옵니다. 여기서 도구는 `retriever` 도구를 가져오게 되며, 이 도구는 `mcp_server_rag.py` 에서 정의된 도구입니다. 이 파일은 사전에 서버에서 실행되지 **않아도** 됩니다.

In [19]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_mcp_adapters.tools import load_mcp_tools
from langgraph.prebuilt import create_react_agent
from langchain_anthropic import ChatAnthropic
from langchain_teddynote.messages import astream_graph

# Anthropic의 Claude 모델 초기화
# model = ChatAnthropic(
#     model_name="claude-3-7-sonnet-latest", temperature=0, max_tokens=20000
# )
model = ChatGoogleGenerativeAI(model="gemini-2.0-flash-exp")

# RAG 서버를 위한 StdIO 서버 파라미터 설정
server_params = StdioServerParameters(
    command="./.venv/bin/python",
    args=["./mcp_server_rag.py"],
)

# StdIO 클라이언트를 사용하여 RAG 서버와 통신
async with stdio_client(server_params) as (read, write):
    # 클라이언트 세션 생성
    async with ClientSession(read, write) as session:
        # 연결 초기화
        await session.initialize()

        # MCP 도구 로드 (여기서는 retriever 도구)
        tools = await load_mcp_tools(session)

        # 에이전트 생성 및 실행
        agent = create_react_agent(model, tools)

        # 에이전트 응답 스트리밍
        await astream_graph(
            agent, {"messages": "삼성전자가 개발한 생성형 AI의 이름을 검색해줘"}
        )


🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
삼성전자, '삼성 AI 포럼'서 자체 개발 생성형 AI '삼성 가우스' 공개
2023/11/08
삼성전자가 8일 삼성전자 서울R&D캠퍼스에서 ‘삼성 AI 포럼 2023’ 둘째 날 행사를 개최했다.
삼성리서치에서 주관한 이날 포럼에는 삼성전자의 AI 연구 인력뿐만 아니라, AI 기술 교류를 위해 학계
및 업계 AI 전문가 150여 명이 참석했다.
삼성전자는 2017년부터 매년 SAIT와 삼성리서치 주관으로 ‘삼성 AI포럼’을 개최하며 AI 핵심기술 발전
방향과 혁신을 논의하고 AI 리더십을 강화하고 있다.
한 자리에 모인 AI 전문가들은 전 세계적인 화두가 되고 있는 생성형 AI 기술의 발전 방향을 논의하고
관련 기술에 대한 최신 동향을 공유했다.
또한, AI 기술에 대한 논의뿐만 아니라 생성형 AI 기술이 발전하면서 인간의 삶이 어떻게 변화할 지에
대한 심도 깊은 논의도 진행했다.
 
삼성전자 자체 개발 생성형 AI 모델 ‘삼성 가우스’ 최초 공개
이번 포럼에서는 삼성리서치에서 개발한 생성형 AI 모델 ‘삼성 가우스(Samsung Gauss)’가 처음으로
공개되어 많은 관심을 받았다.
삼성전자는 ‘삼성 가우스’를 활용해 회사 내 업무 혁신을 추진하고 나아가 사람들의 일상에 새로운
경험을 제공하기 위해 생성형 AI 기술을 발전시킬 계획이다.
‘삼성 가우스’는 정규분포 이론을 정립한 천재 수학자 칼 프리드리히 가우스(Carl Friedrich Gauss)
로부터 영감을 얻은 생성형 AI 모델로, 삼성이 추구하는 생성형 AI의 무한한 가능성을 의미한다.
삼성 가우스는 머신 러닝 기술을 기반으로 ▲텍스트를 생성하는 언어 모델(Samsung Gauss
Language) ▲코드를 생성하는 코드 모델(Samsung Gauss Code

## SSE 방식과 StdIO 방식 혼합 사용

- 파일: `mcp_server_rag.py` 는 StdIO 방식으로 통신
- `langchain-dev-docs` 는 SSE 방식으로 통신

SSE 방식과 StdIO 방식을 혼합하여 사용합니다.

In [21]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langchain_anthropic import ChatAnthropic

# Anthropic의 Claude 모델 초기화
# model = ChatAnthropic(
#     model_name="claude-3-7-sonnet-latest", temperature=0, max_tokens=20000
# )
model = ChatGoogleGenerativeAI(model="gemini-2.0-flash-exp")

# 1. 다중 서버 MCP 클라이언트 생성
client = MultiServerMCPClient(
    {
        "document-retriever": {
            # "url": "http://localhost:8005/sse",
            # SSE(Server-Sent Events) 방식으로 통신
            "command": "python",
            "args": ["mcp_server_rag.py"],
            "transport": "stdio",
        },
        "langchain-dev-docs": {
            # SSE 서버가 8765 포트에서 실행 중인지 확인하세요
            "url": "http://teddynote.io:8765/sse",
            # SSE(Server-Sent Events) 방식으로 통신
            "transport": "sse",
        },
    }
)


# 2. 비동기 컨텍스트 매니저를 통한 명시적 연결 초기화
await client.__aenter__()

langgraph 의 `create_react_agent` 를 사용하여 에이전트를 생성합니다.

In [23]:
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.runnables import RunnableConfig

prompt = (
    "You are a smart agent. "
    "Use `retriever` tool to search on AI related documents and answer questions."
    "Use `langchain-dev-docs` tool to search on langchain / langgraph related documents and answer questions."
    "Answer in Korean."
)
agent = create_react_agent(
    model, client.get_tools(), prompt=prompt, checkpointer=MemorySaver()
)

구축해 놓은 `mcp_server_rag.py` 에서 정의한 `retriever` 도구를 사용하여 검색을 수행합니다.

In [27]:
config = RunnableConfig(recursion_limit=30, thread_id=1)
await astream_graph(
    agent,
    {
        "messages": "`retriever` 도구를 사용해서 삼성전자가 개발한 생성형 AI 이름을 검색해줘"
    },
    config=config,
)


🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
삼성전자, '삼성 AI 포럼'서 자체 개발 생성형 AI '삼성 가우스' 공개
2023/11/08
삼성전자가 8일 삼성전자 서울R&D캠퍼스에서 ‘삼성 AI 포럼 2023’ 둘째 날 행사를 개최했다.
삼성리서치에서 주관한 이날 포럼에는 삼성전자의 AI 연구 인력뿐만 아니라, AI 기술 교류를 위해 학계
및 업계 AI 전문가 150여 명이 참석했다.
삼성전자는 2017년부터 매년 SAIT와 삼성리서치 주관으로 ‘삼성 AI포럼’을 개최하며 AI 핵심기술 발전
방향과 혁신을 논의하고 AI 리더십을 강화하고 있다.
한 자리에 모인 AI 전문가들은 전 세계적인 화두가 되고 있는 생성형 AI 기술의 발전 방향을 논의하고
관련 기술에 대한 최신 동향을 공유했다.
또한, AI 기술에 대한 논의뿐만 아니라 생성형 AI 기술이 발전하면서 인간의 삶이 어떻게 변화할 지에
대한 심도 깊은 논의도 진행했다.
 
삼성전자 자체 개발 생성형 AI 모델 ‘삼성 가우스’ 최초 공개
이번 포럼에서는 삼성리서치에서 개발한 생성형 AI 모델 ‘삼성 가우스(Samsung Gauss)’가 처음으로
공개되어 많은 관심을 받았다.
삼성전자는 ‘삼성 가우스’를 활용해 회사 내 업무 혁신을 추진하고 나아가 사람들의 일상에 새로운
경험을 제공하기 위해 생성형 AI 기술을 발전시킬 계획이다.
‘삼성 가우스’는 정규분포 이론을 정립한 천재 수학자 칼 프리드리히 가우스(Carl Friedrich Gauss)
로부터 영감을 얻은 생성형 AI 모델로, 삼성이 추구하는 생성형 AI의 무한한 가능성을 의미한다.
삼성 가우스는 머신 러닝 기술을 기반으로 ▲텍스트를 생성하는 언어 모델(Samsung Gauss
Language) ▲코드를 생성하는 코드 모델(Samsung Gauss Code

{'node': 'agent',
 'content': AIMessageChunk(content=' 가지 모델로 구성되어 있습니다.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash-exp', 'safety_ratings': []}, id='run-db105659-9be2-48b1-a3c5-be88be9d9cfb', usage_metadata={'input_tokens': 50, 'output_tokens': 47, 'total_tokens': 97, 'input_token_details': {'cache_read': 0}}),
 'metadata': {'thread_id': 1,
  'langgraph_step': 16,
  'langgraph_node': 'agent',
  'langgraph_triggers': ('branch:to:agent', 'start:agent', 'tools'),
  'langgraph_path': ('__pregel_pull', 'agent'),
  'langgraph_checkpoint_ns': 'agent:8b5c7699-3456-32aa-46c1-f83f215741f0',
  'checkpoint_ns': 'agent:8b5c7699-3456-32aa-46c1-f83f215741f0',
  'ls_provider': 'google_genai',
  'ls_model_name': 'models/gemini-2.0-flash-exp',
  'ls_model_type': 'chat',
  'ls_temperature': 0.7}}

이번에는 `langchain-dev-docs` 도구를 사용하여 검색을 수행합니다.

In [25]:
config = RunnableConfig(recursion_limit=30, thread_id=1)
await astream_graph(
    agent,
    {"messages": "langgraph-dev-docs 참고해서 self-rag 의 정의에 대해서 알려줘"},
    config=config,
)


🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
["Below is the list of references for the answer to the question.(Ordered by relevance descending):", "1. https://langchain-ai.github.io/langgraph/tutorials/rag/langgraph_adaptive_rag/", "2. https://langchain-ai.github.io/langgraph/tutorials/rag/langgraph_self_rag/", "3. https://langchain-ai.github.io/langgraph/tutorials/rag/langgraph_crag/", "4. https://langchain-ai.github.io/langgraph/tutorials/code_assistant/langgraph_code_assistant/"]
🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
LangGraph 문서를 참고하여 self-rag의 정의를 찾았습니다. 다음은 관련 문서 URL입니다.

1.  [https://langchain-ai.github.io/langgraph/tutorials/rag/langgraph\_adaptive\_rag/](https://langchain-ai.github.io/langgraph/tutorials/rag/langgraph_adaptive_rag/)
2.  [https://langchain-ai.github.io/langgraph/tutorials/rag/langgraph\_self\_rag/](https://langchain-ai.github.io/langgraph/tuto

{'node': 'agent',
 'content': AIMessageChunk(content=' URL에서 문서를 가져와야 합니다. 어떤 문서를 가져와서 self-rag의 정의를 찾아드릴까요?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash-exp', 'safety_ratings': []}, id='run-c8ae29a6-7a1e-4b75-acba-816eea6a89cd', usage_metadata={'input_tokens': -45, 'output_tokens': 185, 'total_tokens': 140, 'input_token_details': {'cache_read': 0}}),
 'metadata': {'thread_id': 1,
  'langgraph_step': 8,
  'langgraph_node': 'agent',
  'langgraph_triggers': ('branch:to:agent', 'start:agent', 'tools'),
  'langgraph_path': ('__pregel_pull', 'agent'),
  'langgraph_checkpoint_ns': 'agent:4d6f3a91-4e0c-dc4a-274a-995534e5b781',
  'checkpoint_ns': 'agent:4d6f3a91-4e0c-dc4a-274a-995534e5b781',
  'ls_provider': 'google_genai',
  'ls_model_name': 'models/gemini-2.0-flash-exp',
  'ls_model_type': 'chat',
  'ls_temperature': 0.7}}

`MemorySaver` 를 사용하여 단기 기억을 유지합니다. 따라서, multi-turn 대화도 가능합니다.

In [28]:
await astream_graph(
    agent, {"messages": "이전의 내용을 bullet point 로 요약해줘"}, config=config
)


🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
*   삼성전자가 개발한 생성형 AI 이름은 '삼성 가우스(Samsung Gauss)'입니다.
*   '삼성 가우스'는 텍스트, 코드, 이미지를 생성하는 세 가지 모델로 구성되어 있습니다.

{'node': 'agent',
 'content': AIMessageChunk(content=', 이미지를 생성하는 세 가지 모델로 구성되어 있습니다.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash-exp', 'safety_ratings': []}, id='run-b90ee5fc-8280-4304-a531-165f8f11066a', usage_metadata={'input_tokens': 52, 'output_tokens': 53, 'total_tokens': 105, 'input_token_details': {'cache_read': 0}}),
 'metadata': {'thread_id': 1,
  'langgraph_step': 19,
  'langgraph_node': 'agent',
  'langgraph_triggers': ('branch:to:agent', 'start:agent', 'tools'),
  'langgraph_path': ('__pregel_pull', 'agent'),
  'langgraph_checkpoint_ns': 'agent:12ba5f05-04d4-ca9d-dad5-b2b8f614972c',
  'checkpoint_ns': 'agent:12ba5f05-04d4-ca9d-dad5-b2b8f614972c',
  'ls_provider': 'google_genai',
  'ls_model_name': 'models/gemini-2.0-flash-exp',
  'ls_model_type': 'chat',
  'ls_temperature': 0.7}}

## LangChain 에 통합된 도구 + MCP 도구

여기서는 LangChain 에 통합된 도구를 기존의 MCP 로만 이루어진 도구와 함께 사용이 가능한지 테스트 합니다.

In [29]:
from langchain_teddynote.tools.tavily import TavilySearch

# Tavily 검색 도구를 초기화 합니다. (news 타입, 최근 3일 내 뉴스)
tavily = TavilySearch(max_results=3, topic="news", days=3)

# 기존의 MCP 도구와 함께 사용합니다.
tools = client.get_tools() + [tavily]

langgraph 의 `create_react_agent` 를 사용하여 에이전트를 생성합니다.

In [30]:
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.runnables import RunnableConfig

prompt = "You are a smart agent with various tools. Answer questions in Korean."
agent = create_react_agent(model, tools, prompt=prompt, checkpointer=MemorySaver())

새롭게 추가한 `tavily` 도구를 사용하여 검색을 수행합니다.

In [31]:
await astream_graph(agent, {"messages": "오늘 뉴스 찾아줘"}, config=config)


🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
어떤 주제에 대한 뉴스를 찾으시나요? 구체적인 주제를 알려주시면 더 정확한 결과를 얻을 수 있습니다. 예를 들어 "오늘 날씨", "오늘 주식 시장", 또는 특정 회사나 인물에 대한 뉴스를 찾을 수 있습니다.

{'node': 'agent',
 'content': AIMessageChunk(content=' 날씨", "오늘 주식 시장", 또는 특정 회사나 인물에 대한 뉴스를 찾을 수 있습니다.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash-exp', 'safety_ratings': []}, id='run-23494c31-b431-45d2-abb4-5f3392faef35', usage_metadata={'input_tokens': -120, 'output_tokens': 71, 'total_tokens': -49, 'input_token_details': {'cache_read': 0}}),
 'metadata': {'thread_id': 1,
  'langgraph_step': 1,
  'langgraph_node': 'agent',
  'langgraph_triggers': ('branch:to:agent', 'start:agent', 'tools'),
  'langgraph_path': ('__pregel_pull', 'agent'),
  'langgraph_checkpoint_ns': 'agent:5c2865fc-a7ec-7dd0-ca47-5ab444ab077b',
  'checkpoint_ns': 'agent:5c2865fc-a7ec-7dd0-ca47-5ab444ab077b',
  'ls_provider': 'google_genai',
  'ls_model_name': 'models/gemini-2.0-flash-exp',
  'ls_model_type': 'chat',
  'ls_temperature': 0.7}}

`retriever` 도구가 원활하게 작동하는 것을 확인할 수 있습니다.

In [32]:
await astream_graph(
    agent,
    {
        "messages": "`retriever` 도구를 사용해서 삼성전자가 개발한 생성형 AI 이름을 검색해줘"
    },
    config=config,
)


🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
삼성전자, '삼성 AI 포럼'서 자체 개발 생성형 AI '삼성 가우스' 공개
2023/11/08
삼성전자가 8일 삼성전자 서울R&D캠퍼스에서 ‘삼성 AI 포럼 2023’ 둘째 날 행사를 개최했다.
삼성리서치에서 주관한 이날 포럼에는 삼성전자의 AI 연구 인력뿐만 아니라, AI 기술 교류를 위해 학계
및 업계 AI 전문가 150여 명이 참석했다.
삼성전자는 2017년부터 매년 SAIT와 삼성리서치 주관으로 ‘삼성 AI포럼’을 개최하며 AI 핵심기술 발전
방향과 혁신을 논의하고 AI 리더십을 강화하고 있다.
한 자리에 모인 AI 전문가들은 전 세계적인 화두가 되고 있는 생성형 AI 기술의 발전 방향을 논의하고
관련 기술에 대한 최신 동향을 공유했다.
또한, AI 기술에 대한 논의뿐만 아니라 생성형 AI 기술이 발전하면서 인간의 삶이 어떻게 변화할 지에
대한 심도 깊은 논의도 진행했다.
 
삼성전자 자체 개발 생성형 AI 모델 ‘삼성 가우스’ 최초 공개
이번 포럼에서는 삼성리서치에서 개발한 생성형 AI 모델 ‘삼성 가우스(Samsung Gauss)’가 처음으로
공개되어 많은 관심을 받았다.
삼성전자는 ‘삼성 가우스’를 활용해 회사 내 업무 혁신을 추진하고 나아가 사람들의 일상에 새로운
경험을 제공하기 위해 생성형 AI 기술을 발전시킬 계획이다.
‘삼성 가우스’는 정규분포 이론을 정립한 천재 수학자 칼 프리드리히 가우스(Carl Friedrich Gauss)
로부터 영감을 얻은 생성형 AI 모델로, 삼성이 추구하는 생성형 AI의 무한한 가능성을 의미한다.
삼성 가우스는 머신 러닝 기술을 기반으로 ▲텍스트를 생성하는 언어 모델(Samsung Gauss
Language) ▲코드를 생성하는 코드 모델(Samsung Gauss Code

{'node': 'agent',
 'content': AIMessageChunk(content='.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash-exp', 'safety_ratings': []}, id='run-b49fb851-d834-4f4d-8182-f19fc5280a6c', usage_metadata={'input_tokens': -34, 'output_tokens': 23, 'total_tokens': -11, 'input_token_details': {'cache_read': 0}}),
 'metadata': {'thread_id': 1,
  'langgraph_step': 6,
  'langgraph_node': 'agent',
  'langgraph_triggers': ('branch:to:agent', 'start:agent', 'tools'),
  'langgraph_path': ('__pregel_pull', 'agent'),
  'langgraph_checkpoint_ns': 'agent:b67b1574-1961-ef1f-2fa3-028f6ac2af31',
  'checkpoint_ns': 'agent:b67b1574-1961-ef1f-2fa3-028f6ac2af31',
  'ls_provider': 'google_genai',
  'ls_model_name': 'models/gemini-2.0-flash-exp',
  'ls_model_type': 'chat',
  'ls_temperature': 0.7}}

## Smithery 에서 제공하는 MCP 서버

- 링크: https://smithery.ai/

사용한 도구 목록은 아래와 같습니다.

- Sequential Thinking: https://smithery.ai/server/@smithery-ai/server-sequential-thinking
  - 구조화된 사고 프로세스를 통해 역동적이고 성찰적인 문제 해결을 위한 도구를 제공하는 MCP 서버
- Desktop Commander: https://smithery.ai/server/@wonderwhy-er/desktop-commander
  - 다양한 편집 기능으로 터미널 명령을 실행하고 파일을 관리하세요. 코딩, 셸 및 터미널, 작업 자동화

**참고**

- smithery 에서 제공하는 도구를 JSON 형식으로 가져올때, 아래의 예시처럼 `"transport": "stdio"` 로 꼭 설정해야 합니다.

In [18]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langchain_anthropic import ChatAnthropic

# LLM 모델 초기화
model = ChatAnthropic(model="claude-3-7-sonnet-latest", temperature=0, max_tokens=20000)

# 1. 클라이언트 생성
client = MultiServerMCPClient(
    {
        "server-sequential-thinking": {
            "command": "npx",
            "args": [
                "-y",
                "@smithery/cli@latest",
                "run",
                "@smithery-ai/server-sequential-thinking",
                "--key",
                "89a4780a-53b7-4b7b-92e9-a29815f2669b",
            ],
            "transport": "stdio",  # stdio 방식으로 통신을 추가합니다.
        },
        "desktop-commander": {
            "command": "npx",
            "args": [
                "-y",
                "@smithery/cli@latest",
                "run",
                "@wonderwhy-er/desktop-commander",
                "--key",
                "89a4780a-53b7-4b7b-92e9-a29815f2669b",
            ],
            "transport": "stdio",  # stdio 방식으로 통신을 추가합니다.
        },
        "document-retriever": {
            "command": "./.venv/bin/python",
            # mcp_server_rag.py 파일의 절대 경로로 업데이트해야 합니다
            "args": ["./mcp_server_rag.py"],
            # stdio 방식으로 통신 (표준 입출력 사용)
            "transport": "stdio",
        },
    }
)


# 2. 명시적으로 연결 초기화
await client.__aenter__()

langgraph 의 `create_react_agent` 를 사용하여 에이전트를 생성합니다.

In [19]:
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.runnables import RunnableConfig


agent = create_react_agent(model, client.get_tools(), checkpointer=MemorySaver())

`Desktop Commander` 도구를 사용하여 터미널 명령을 실행합니다.

In [20]:
await astream_graph(
    agent,
    {
        "messages": "현재 경로를 포함한 하위 폴더 구조를 tree 로 그려줘. 단, .venv 폴더는 제외하고 출력해줘."
    },
    config=config,
)


🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
현재 경로와 하위 폴더 구조를 tree 형태로 그려드리겠습니다. `.venv` 폴더는 제외하고 출력하겠습니다.

이를 위해 `execute_command` 함수를 사용하여 tree 명령어를 실행하겠습니다. `-I` 옵션을 사용하면 특정 패턴을 제외할 수 있습니다.
🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
Command started with PID 5105
Initial output:
.
|____.git
| |____hooks
| |____info
| |____logs
| | |____refs
| | | |____heads
| | | |____remotes
| | | | |____origin
| |____objects
| | |____03
| | |____18
| | |____2d
| | |____36
| | |____3c
| | |____42
| | |____48
| | |____4d
| | |____59
| | |____5d
| | |____6f
| | |____75
| | |____93
| | |____96
| | |____99
| | |____9f
| | |____a7
| | |____a9
| | |____aa
| | |____b5
| | |____c0
| | |____c1
| | |____c8
| | |____cb
| | |____e2
| | |____e3
| | |____e4
| | |____e7
| | |____ea
| | |____f0
| | |____f7
| | |____info
| | |____pack
| |____refs
| | |____heads
| | |____remotes
| | | |____origin
| | |____tags
|____assets
|____data

🔄 Node: agent 🔄
- - - - - - - - - - - 

{'node': 'agent',
 'content': AIMessageChunk(content='', additional_kwargs={}, response_metadata={'stop_reason': 'end_turn', 'stop_sequence': None}, id='run-7325ff0e-cf97-471f-96ef-0ab2fc0ba215', usage_metadata={'input_tokens': 0, 'output_tokens': 321, 'total_tokens': 321, 'input_token_details': {}}),
 'metadata': {'thread_id': 1,
  'langgraph_step': 9,
  'langgraph_node': 'agent',
  'langgraph_triggers': ('branch:to:agent', 'start:agent', 'tools'),
  'langgraph_path': ('__pregel_pull', 'agent'),
  'langgraph_checkpoint_ns': 'agent:94e82fea-342b-2f07-08b0-432160e4029c',
  'checkpoint_ns': 'agent:94e82fea-342b-2f07-08b0-432160e4029c',
  'ls_provider': 'anthropic',
  'ls_model_name': 'claude-3-7-sonnet-latest',
  'ls_model_type': 'chat',
  'ls_temperature': 0.0,
  'ls_max_tokens': 20000}}

이번에는 `Sequential Thinking` 도구를 사용하여 비교적 복잡한 작업을 수행할 수 있는지 확인합니다.

In [21]:
await astream_graph(
    agent,
    {
        "messages": (
            "`retriever` 도구를 사용해서 삼성전자가 개발한 생성형 AI 관련 내용을 검색하고 "
            "`Sequential Thinking` 도구를 사용해서 보고서를 작성해줘."
        )
    },
    config=config,
)


🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
삼성전자가 개발한 생성형 AI 관련 내용을 검색하고 보고서를 작성해드리겠습니다. 먼저 `retrieve` 도구를 사용하여 관련 정보를 검색한 후, `sequentialthinking` 도구를 활용해 체계적인 보고서를 작성하겠습니다.
🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
SPRi AI Brief |  
2023-12월호
10
삼성전자, 자체 개발 생성 AI ‘삼성 가우스’ 공개
n 삼성전자가 온디바이스에서 작동 가능하며 언어, 코드, 이미지의 3개 모델로 구성된 자체 개발 생성 
AI 모델 ‘삼성 가우스’를 공개
n 삼성전자는 삼성 가우스를 다양한 제품에 단계적으로 탑재할 계획으로, 온디바이스 작동이 가능한 
삼성 가우스는 외부로 사용자 정보가 유출될 위험이 없다는 장점을 보유
KEY Contents
£ 언어, 코드, 이미지의 3개 모델로 구성된 삼성 가우스, 온디바이스 작동 지원
n 삼성전자가 2023년 11월 8일 열린 ‘삼성 AI 포럼 2023’ 행사에서 자체 개발한 생성 AI 모델 
‘삼성 가우스’를 최초 공개
∙정규분포 이론을 정립한 천재 수학자 가우스(Gauss)의 이름을 본뜬 삼성 가우스는 다양한 상황에 
최적화된 크기의 모델 선택이 가능
∙삼성 가우스는 라이선스나 개인정보를 침해하지 않는 안전한 데이터를 통해 학습되었으며, 
온디바이스에서 작동하도록 설계되어 외부로 사용자의 정보가 유출되지 않는 장점을 보유
∙삼성전자는 삼성 가우스를 활용한 온디바이스 AI 기술도 소개했으며, 생성 AI 모델을 다양한 제품에 
단계적으로 탑재할 계획
n 삼성 가우스는 △텍스트를 생성하는 언어모델 △코드를 생성하는 코드 모델 △이미지를 생성하는 
이미지 모델의 3개 모델로 구성
∙언어 모델은 클라우드와 온디바이스 대상 다양한 모델로 구성되며, 메일 작성, 문서 요약,

{'node': 'agent',
 'content': AIMessageChunk(content='', additional_kwargs={}, response_metadata={'stop_reason': 'end_turn', 'stop_sequence': None}, id='run-467bb435-0acf-42a3-8d47-601b8775393a', usage_metadata={'input_tokens': 0, 'output_tokens': 2247, 'total_tokens': 2247, 'input_token_details': {}}),
 'metadata': {'thread_id': 1,
  'langgraph_step': 32,
  'langgraph_node': 'agent',
  'langgraph_triggers': ('branch:to:agent', 'start:agent', 'tools'),
  'langgraph_path': ('__pregel_pull', 'agent'),
  'langgraph_checkpoint_ns': 'agent:8e9e85a9-cd93-4316-29f0-663a875bec41',
  'checkpoint_ns': 'agent:8e9e85a9-cd93-4316-29f0-663a875bec41',
  'ls_provider': 'anthropic',
  'ls_model_name': 'claude-3-7-sonnet-latest',
  'ls_model_type': 'chat',
  'ls_temperature': 0.0,
  'ls_max_tokens': 20000}}